# Fraud Detection in Financial Transactions

## Overview
This notebook demonstrates the implementation of machine learning algorithms to identify potentially fraudulent transactions in financial systems. The analysis focuses on enhancing security measures and reducing financial losses due to fraud through advanced detection techniques.

## Dataset
We'll be using the [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset from Kaggle, which contains transactions made by credit cards in September 2013 by European cardholders. This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve, auc, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set(style='whitegrid')
plt.style.use('seaborn-whitegrid')
%matplotlib inline

In [2]:
# Load the dataset
df = pd.read_csv('creditcard.csv')

# Display the first few rows
df.head()

## Data Exploration and Preprocessing

In [3]:
# Check basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()
print("\nSummary Statistics:")
df.describe()

In [4]:
# Check for missing values
print("Missing Values:")
df.isnull().sum()

In [5]:
# Check class distribution
class_distribution = df['Class'].value_counts()
print("Class Distribution:")
print(class_distribution)
print(f"Percentage of Fraud Cases: {class_distribution[1]/len(df)*100:.4f}%")

In [6]:
# Visualize class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='Class', data=df)
plt.title('Class Distribution (0: Normal, 1: Fraud)', fontsize=15)
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.yscale('log')  # Using log scale due to class imbalance
plt.grid(True)
plt.show()

In [7]:
# Analyze transaction amount by class
plt.figure(figsize=(12, 6))
sns.boxplot(x='Class', y='Amount', data=df)
plt.title('Transaction Amount by Class', fontsize=15)
plt.xlabel('Class (0: Normal, 1: Fraud)', fontsize=12)
plt.ylabel('Amount', fontsize=12)
plt.grid(True)
plt.show()

In [8]:
# Distribution of transaction amount for fraudulent transactions
plt.figure(figsize=(12, 6))
sns.histplot(df[df['Class'] == 1]['Amount'], kde=True)
plt.title('Distribution of Transaction Amount for Fraudulent Transactions', fontsize=15)
plt.xlabel('Amount', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(True)
plt.show()

In [9]:
# Correlation matrix for feature analysis
plt.figure(figsize=(20, 16))
correlation_matrix = df.corr()
mask = np.triu(correlation_matrix)
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix', fontsize=15)
plt.show()

## Feature Engineering and Data Preparation

In [10]:
# Standardize the 'Amount' and 'Time' features
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop the original 'Amount' and 'Time' columns
df = df.drop(['Amount', 'Time'], axis=1)

# Display the updated dataframe
df.head()

In [11]:
# Split the data into features and target variable
X = df.drop('Class', axis=1)
y = df['Class']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Training set fraud cases: {sum(y_train)}")
print(f"Testing set fraud cases: {sum(y_test)}")

## Handling Class Imbalance with SMOTE

In [12]:
# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"Resampled training set shape: {X_train_smote.shape}")
print(f"Original training set class distribution: {pd.Series(y_train).value_counts()}")
print(f"Resampled training set class distribution: {pd.Series(y_train_smote).value_counts()}")

## Model Training and Evaluation

In [13]:
# Function to evaluate model performance
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate evaluation metrics
    print(f"\n{model_name} - Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{model_name} - Confusion Matrix', fontsize=15)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.show()
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'{model_name} - ROC Curve', fontsize=15)
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()
    
    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = auc(recall, precision)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (area = {pr_auc:.2f})')
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'{model_name} - Precision-Recall Curve', fontsize=15)
    plt.legend(loc="lower left")
    plt.grid(True)
    plt.show()
    
    return model, roc_auc, pr_auc

### Logistic Regression

In [14]:
# Train and evaluate Logistic Regression model
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model, lr_roc_auc, lr_pr_auc = evaluate_model(lr_model, X_train_smote, y_train_smote, X_test, y_test, "Logistic Regression")

### Random Forest

In [15]:
# Train and evaluate Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model, rf_roc_auc, rf_pr_auc = evaluate_model(rf_model, X_train_smote, y_train_smote, X_test, y_test, "Random Forest")

### XGBoost

In [16]:
# Train and evaluate XGBoost model
xgb_model = xgb.XGBClassifier(scale_pos_weight=len(y_train_smote) / sum(y_train_smote), random_state=42)
xgb_model, xgb_roc_auc, xgb_pr_auc = evaluate_model(xgb_model, X_train_smote, y_train_smote, X_test, y_test, "XGBoost")

## Feature Importance Analysis

In [17]:
# Analyze feature importance from Random Forest model
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(15))
plt.title('Top 15 Feature Importance from Random Forest', fontsize=15)
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True)
plt.show()

## Model Comparison

In [18]:
# Compare model performance
models = ['Logistic Regression', 'Random Forest', 'XGBoost']
roc_aucs = [lr_roc_auc, rf_roc_auc, xgb_roc_auc]
pr_aucs = [lr_pr_auc, rf_pr_auc, xgb_pr_auc]

comparison_df = pd.DataFrame({
    'Model': models,
    'ROC AUC': roc_aucs,
    'PR AUC': pr_aucs
})

print("Model Performance Comparison:")
print(comparison_df)

# Visualize model comparison
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.barplot(x='Model', y='ROC AUC', data=comparison_df)
plt.title('ROC AUC Comparison', fontsize=15)
plt.ylim(0.9, 1.0)  # Adjust y-axis for better visualization
plt.grid(True)

plt.subplot(1, 2, 2)
sns.barplot(x='Model', y='PR AUC', data=comparison_df)
plt.title('PR AUC Comparison', fontsize=15)
plt.ylim(0.5, 1.0)  # Adjust y-axis for better visualization
plt.grid(True)

plt.tight_layout()
plt.show()

## Threshold Optimization for Best Model

In [19]:
# Determine the best model based on PR AUC (more relevant for imbalanced data)
best_model_index = comparison_df['PR AUC'].idxmax()
best_model_name = comparison_df.loc[best_model_index, 'Model']
print(f"Best model based on PR AUC: {best_model_name}")

# Select the best model
if best_model_name == 'Logistic Regression':
    best_model = lr_model
elif best_model_name == 'Random Forest':
    best_model = rf_model
else:  # XGBoost
    best_model = xgb_model

# Get prediction probabilities
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Find optimal threshold
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * recall * precision / (recall + precision + 1e-10)  # Adding small epsilon to avoid division by zero
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal threshold: {optimal_threshold:.4f}")
print(f"At optimal threshold - Precision: {precision[optimal_idx]:.4f}, Recall: {recall[optimal_idx]:.4f}, F1: {f1_scores[optimal_idx]:.4f}")

In [20]:
# Apply optimal threshold to predictions
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

# Evaluate with optimal threshold
print(f"\n{best_model_name} with Optimal Threshold - Classification Report:")
print(classification_report(y_test, y_pred_optimal))

# Confusion Matrix with optimal threshold
cm_optimal = confusion_matrix(y_test, y_pred_optimal)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_optimal, annot=True, fmt='d', cmap='Blues')
plt.title(f'{best_model_name} with Optimal Threshold - Confusion Matrix', fontsize=15)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()

## Cost-Benefit Analysis

In [21]:
# Define cost matrix
# Assuming average fraud amount is $100, and cost of investigating a transaction is $10
avg_fraud_amount = 100
investigation_cost = 10

# Cost matrix
# True Negative: No cost
# False Positive: Cost of investigation
# False Negative: Cost of fraud
# True Positive: Cost of investigation (but saved the fraud amount)

# Calculate costs for different thresholds
thresholds_to_test = np.linspace(0.01, 0.99, 99)
costs = []

for threshold in thresholds_to_test:
    y_pred_t = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    
    cost = fp * investigation_cost + fn * avg_fraud_amount
    costs.append(cost)

# Find threshold with minimum cost
min_cost_idx = np.argmin(costs)
min_cost_threshold = thresholds_to_test[min_cost_idx]

print(f"Minimum cost threshold: {min_cost_threshold:.4f}")
print(f"Minimum cost: ${costs[min_cost_idx]:.2f}")

# Plot cost vs threshold
plt.figure(figsize=(10, 6))
plt.plot(thresholds_to_test, costs)
plt.axvline(x=min_cost_threshold, color='r', linestyle='--', label=f'Min Cost Threshold: {min_cost_threshold:.4f}')
plt.title('Cost vs Threshold', fontsize=15)
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Cost ($)', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

## Real-time Fraud Detection System Design

Based on our analysis, we can design a real-time fraud detection system with the following components:

1. **Data Preprocessing Pipeline**:
   - Feature standardization (especially for Amount and Time)
   - Feature engineering based on domain knowledge
   - Real-time data validation and cleaning

2. **Model Deployment**:
   - Deploy the best model (XGBoost or Random Forest) as a microservice
   - Implement A/B testing framework for model comparison
   - Set up model versioning and monitoring

3. **Threshold Management**:
   - Implement dynamic thresholding based on cost-benefit analysis
   - Allow for different thresholds based on transaction characteristics
   - Regularly update thresholds based on new data

4. **Alert System**:
   - High-priority alerts for high-confidence fraud predictions
   - Integration with case management system
   - Automated actions for certain fraud patterns

5. **Feedback Loop**:
   - Capture investigation outcomes for model retraining
   - Implement active learning for ambiguous cases
   - Regular model retraining with new labeled data

6. **Monitoring and Reporting**:
   - Real-time dashboards for fraud detection metrics
   - Anomaly detection for model performance degradation
   - Regular reports on fraud patterns and trends

## Conclusion

This fraud detection analysis has demonstrated the effectiveness of machine learning algorithms in identifying potentially fraudulent transactions in financial systems. We have addressed the challenge of class imbalance using SMOTE and evaluated multiple models to find the best performer.

Key findings include:
- The dataset exhibits extreme class imbalance with only 0.17% fraudulent transactions
- XGBoost and Random Forest models outperform Logistic Regression in fraud detection
- Optimizing the classification threshold significantly improves model performance
- Cost-benefit analysis provides a practical approach to threshold selection
- Feature importance analysis reveals the most predictive variables for fraud detection

These insights can be used to implement a robust fraud detection system that balances the cost of investigation with the potential losses from fraud. By leveraging machine learning techniques and domain knowledge, financial institutions can enhance their security measures and reduce financial losses due to fraudulent activities.

Future work could include:
- Incorporating temporal features and sequence modeling
- Exploring deep learning approaches such as autoencoders for anomaly detection
- Implementing ensemble methods combining multiple models
- Developing explainable AI techniques to interpret model decisions
- Integrating network analysis to identify fraud rings and coordinated attacks